# Multi-task GNN for spinel Li-ion cathode discovery: voltage + thermal stability co-prediction

_Canonical workflow code is real; `# TODO` markers indicate where customization is required._


## Hypothesis

A multi-task graph neural network trained on Materials Project structural and computed-voltage data, with auxiliary heads for thermal decomposition temperature and energy-above-hull, will identify spinel Li-ion cathode candidates that achieve a top-20 synthesis hit-rate ≥ 30% on a closed-loop experimental validation cohort, where 'hit' means measured voltage > 4.0V AND decomposition onset > 180°C.

### Falsifiability
- **Prediction:** The multi-task GNN screen achieves a higher top-20 synthesis hit-rate on a held-out spinel discovery cohort than single-task voltage-only baselines and a hull-distance-only filter.
- **Threshold:** Top-20 hit-rate >= 30% (where 'hit' = measured voltage > 4.0V AND decomp onset > 180°C), compared to baseline hit-rate that must be empirically established but is expected near 10-15% from random spinel selection.
- **Null outcome:** Top-20 hit-rate < 15% falsifies the hypothesis: the multi-task screen is no better than random spinel selection from the MP catalog.


In [ ]:
# === Imports — materials informatics stack ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pymatgen.core import Structure, Composition
from pymatgen.ext.matproj import MPRester
# from matminer.featurizers.composition import ElementProperty
# from matminer.featurizers.structure import SiteStatsFingerprint

from sklearn.ensemble import RandomForestRegressor, GradientBoostingClassifier
from sklearn.metrics import mean_absolute_error, r2_score, top_k_accuracy_score
from sklearn.model_selection import KFold

import torch
import torch.nn as nn
# import torch_geometric as pyg

RANDOM_SEED = 0
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

MP_API_KEY = None  # TODO: set your https://materialsproject.org API key


## Data acquisition

- **Primary dataset:** Materials Project full catalog (filtered to oxide and oxofluoride spinels and spinel-adjacent space groups)
- **Accession / URL:** `https://materialsproject.org via pymatgen.ext.matproj API`
- **Access constraints:** free, requires MP API key (registration only)


In [ ]:
# === Data acquisition (Materials Project canonical pattern) ===
PRIMARY_REFERENCE = 'https://materialsproject.org via pymatgen.ext.matproj API'

if MP_API_KEY is None:
    raise NotImplementedError(
        f'Set MP_API_KEY before continuing. Reference: {PRIMARY_REFERENCE}'
    )

# TODO: tune the search criteria for your hypothesis. Spinel example:
FORMULA_PATTERN = '*Mn2O4'  # TODO: replace with your composition family
FIELDS = [
    'material_id', 'formula_pretty', 'structure',
    'formation_energy_per_atom', 'energy_above_hull',
    'band_gap', 'is_stable', 'theoretical',
]

with MPRester(MP_API_KEY) as mpr:
    docs = mpr.summary.search(formula=FORMULA_PATTERN, fields=FIELDS)

df = pd.DataFrame([{f: getattr(d, f, None) for f in FIELDS} for d in docs])
print(f'queried: {len(df)} candidates matching {FORMULA_PATTERN!r}')
print(df[['formula_pretty', 'energy_above_hull', 'band_gap', 'is_stable']].head())


## Step 1: Spinel subset extraction

Query MP for entries with prototype spinel topology; supplement with spinel-adjacent space groups.

**Methods cited:**
- github.com/materialsproject/pymatgen


In [ ]:
# === Step 1: Spinel subset extraction ===
# Filter the queried set to your structural family of interest.

MAX_HULL_DISTANCE_EV = 0.05  # TODO: tune; 0 = thermodynamically stable, 0.1 = mildly metastable
REQUIRE_STABLE = False        # TODO: True for stable-only, False to include metastables

df_filt = df.copy()
df_filt = df_filt[df_filt['energy_above_hull'].fillna(99) <= MAX_HULL_DISTANCE_EV]
if REQUIRE_STABLE:
    df_filt = df_filt[df_filt['is_stable'].fillna(False)]
print(f'after stability filter: {len(df_filt)}/{len(df)} retained')

# TODO: add structure-prototype filter for your topology (e.g., spinel space group Fd-3m)
# from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
# df_filt['spacegroup'] = df_filt['structure'].apply(
#     lambda s: SpacegroupAnalyzer(s).get_space_group_symbol() if s else None
# )
# df_filt = df_filt[df_filt['spacegroup'] == 'Fd-3m']


## Step 2: Decomp-temp label harvesting

Mine experimental DSC/TGA decomp temps from thermochemistry tables and DFT-MD literature.

**Methods cited:**
- NIST WebBook


In [ ]:
# === Step 2: Decomp-temp label harvesting ===
# Harvest decomposition / voltage / other property labels.

# Voltage labels via MP intercalation electrode endpoints (canonical pattern):
# with MPRester(MP_API_KEY) as mpr:
#     batt = mpr.materials.insertion_electrodes.search(
#         working_ion='Li', formula='*Mn2O4',
#         fields=['battery_id', 'average_voltage', 'energy_grav', 'capacity_grav'],
#     )

# TODO: harvest decomposition-temperature labels from open thermochemistry
# tables (NIST WebBook) or DFT-MD literature. This is the bottleneck data step.
decomp_temps = pd.DataFrame()  # TODO: populate with material_id → decomp_C
if decomp_temps.empty:
    raise NotImplementedError(
        'Populate decomp_temps with experimental or DFT-MD decomposition labels.'
    )

df_filt = df_filt.merge(decomp_temps, on='material_id', how='left')
print(f'labeled: {df_filt["decomp_C"].notna().sum()}/{len(df_filt)} have decomp temps')


## Step 3: Featurization

Crystal graph + Voronoi neighbors via pymatgen; Goldschmidt + Hume-Rothery features.

**Methods cited:**
- github.com/materialsproject/pymatgen
- Xie & Grossman 2018, Phys Rev Lett 120:145301


In [ ]:
# === Step 3: Featurization ===
# Canonical matminer featurization: composition-level descriptors.
from matminer.featurizers.composition import ElementProperty

ep = ElementProperty.from_preset('magpie')
df_filt['composition'] = df_filt['formula_pretty'].apply(Composition)
feat = ep.featurize_dataframe(df_filt, col_id='composition', ignore_errors=True)
feature_cols = [c for c in feat.columns if c.startswith('MagpieData')]
X = feat[feature_cols].fillna(0).values
print(f'featurized: X shape = {X.shape}')

# TODO: add classical empirical-rule features alongside Magpie:
# - Goldschmidt tolerance factor for perovskite/spinel topologies
# - Hume-Rothery ionic-size mismatch
# - mean electronegativity difference


## Step 4: Multi-task GNN training

M3GNet-class encoder + 3-head MLP with task-weighted MSE.

**Methods cited:**
- Chen & Ong 2022, Nat Comput Sci 2:718


In [ ]:
# === Step 4: Multi-task GNN training ===
# Canonical matminer featurization: composition-level descriptors.
from matminer.featurizers.composition import ElementProperty

ep = ElementProperty.from_preset('magpie')
df_filt['composition'] = df_filt['formula_pretty'].apply(Composition)
feat = ep.featurize_dataframe(df_filt, col_id='composition', ignore_errors=True)
feature_cols = [c for c in feat.columns if c.startswith('MagpieData')]
X = feat[feature_cols].fillna(0).values
print(f'featurized: X shape = {X.shape}')

# TODO: add classical empirical-rule features alongside Magpie:
# - Goldschmidt tolerance factor for perovskite/spinel topologies
# - Hume-Rothery ionic-size mismatch
# - mean electronegativity difference


## Step 5: Candidate ranking

Apply trained model to extended composition space; rank by joint posterior over thresholds.


In [ ]:
# === Step 5: Candidate ranking ===
# Multi-task regression baseline: random-forest per target, then upgrade to GNN.

from sklearn.ensemble import RandomForestRegressor

y_voltage = df_filt.get('average_voltage', pd.Series([np.nan] * len(df_filt))).values
y_decomp  = df_filt.get('decomp_C', pd.Series([np.nan] * len(df_filt))).values
y_hull    = df_filt['energy_above_hull'].values

mask_v = ~np.isnan(y_voltage)
mask_d = ~np.isnan(y_decomp)

models = {}
if mask_v.sum() > 20:
    models['voltage'] = RandomForestRegressor(n_estimators=200, random_state=RANDOM_SEED).fit(X[mask_v], y_voltage[mask_v])
    print(f'voltage model trained on {mask_v.sum()} samples')
if mask_d.sum() > 20:
    models['decomp'] = RandomForestRegressor(n_estimators=200, random_state=RANDOM_SEED).fit(X[mask_d], y_decomp[mask_d])
    print(f'decomp model trained on {mask_d.sum()} samples')
models['hull'] = RandomForestRegressor(n_estimators=200, random_state=RANDOM_SEED).fit(X, y_hull)
print(f'hull model trained on {len(X)} samples')

# TODO: upgrade to a GNN once the RF baseline is established.
# class CrystalGNN(nn.Module):
#     def __init__(self, ...): ...  # TODO: wire CGCNN or M3GNet-class encoder


## Step 6: Closed-loop synthesis + characterization

Synthesize top-20 with experimental partner; measure voltage (galvanostatic) + decomp temp (TGA).


In [ ]:
# === Step 6: Closed-loop synthesis + characterization ===
# Harvest decomposition / voltage / other property labels.

# Voltage labels via MP intercalation electrode endpoints (canonical pattern):
# with MPRester(MP_API_KEY) as mpr:
#     batt = mpr.materials.insertion_electrodes.search(
#         working_ion='Li', formula='*Mn2O4',
#         fields=['battery_id', 'average_voltage', 'energy_grav', 'capacity_grav'],
#     )

# TODO: harvest decomposition-temperature labels from open thermochemistry
# tables (NIST WebBook) or DFT-MD literature. This is the bottleneck data step.
decomp_temps = pd.DataFrame()  # TODO: populate with material_id → decomp_C
if decomp_temps.empty:
    raise NotImplementedError(
        'Populate decomp_temps with experimental or DFT-MD decomposition labels.'
    )

df_filt = df_filt.merge(decomp_temps, on='material_id', how='left')
print(f'labeled: {df_filt["decomp_C"].notna().sum()}/{len(df_filt)} have decomp temps')


## Falsifiability check


In [ ]:
# === Falsifiability check ===
# Primary metric: Top-20 synthesis hit-rate (fraction of top-20 candidates with measured V > 4.0V AND decomp > 180°C)
# Success threshold: Top-20 hit-rate >= 30% on held-out discovery cohort, compared to <15% expected from random spinel selection
# Null outcome:     Top-20 hit-rate < 15% falsifies the hypothesis

try:
    model_metric_value = float(model_hit_rate)
    baseline_metric_value = float(BASELINE_HIT_RATE)
except NameError:
    model_metric_value = None
    baseline_metric_value = None

if model_metric_value is None or baseline_metric_value is None:
    raise NotImplementedError('Run the evaluation step first, or set values manually.')

lift = model_metric_value - baseline_metric_value
print(f'metric (model):    {model_metric_value:.4f}')
print(f'metric (baseline): {baseline_metric_value:.4f}')
print(f'lift:              {lift:+.4f}')

MIN_LIFT_FOR_HYPOTHESIS = 0.15  # TODO: align with falsifiability threshold
assert lift >= MIN_LIFT_FOR_HYPOTHESIS, (
    f'Lift {lift:+.4f} below threshold {MIN_LIFT_FOR_HYPOTHESIS} — hypothesis falsified.'
)
print('falsifiability check PASSED')
